# Adhan PyTorch/Colab Workspace — Scaffold Only

**Scope of this notebook:** environment setup, corpus mount, and loading
one real batch. **No model code, no training loop** — that starts in a
follow-up notebook once this scaffold is proven working.

**Done when:** the last cell below prints a batch shape loaded from a
real `train.bin` produced by `scripts/prepare_slm_corpus.py`, on a free
Colab runtime.

This reuses the repo's own code (cloned fresh below) rather than
duplicating logic here: `src/adhan_slm/data/packing.py`'s
`read_shard()` / `load_manifest()` — the manifest-driven reader that
matches exactly how `prepare_slm_corpus.py` actually writes `train.bin`
(see the note in the batch-loading cell below on why this notebook
does **not** use `scripts/train_efficient.py`'s `MemmapShardDataset` for
this).

## 1. Clone the repo (idempotent)

Gives us `src/adhan_slm/` without duplicating any of its code into this
notebook — if it changes on `main`, re-running this cell picks up the
change.

In [1]:
import os

REPO_DIR = "/content/adhan"

if os.path.isdir(REPO_DIR):
    print("Repo already present, pulling latest ...")
    !cd {REPO_DIR} && git pull
else:
    print("Cloning adhan (shallow) ...")
    !git clone --depth 1 https://github.com/yazhi-lem/adhan.git {REPO_DIR}


Cloning adhan (shallow) ...
Cloning into '/content/adhan'...
remote: Enumerating objects: 189, done.
remote: Counting objects: 100% (189/189), done.
remote: Compressing objects: 100% (174/174), done.
remote: Total 189 (delta 4), reused 124 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (189/189), 651.61 KiB | 899.00 KiB/s, done.
Resolving deltas: 100% (4/4), done.


## 2. Environment check (pinned, not assumed)

Colab's free runtime already ships `torch` matched to whatever GPU/CUDA
driver it handed you for this session — force-reinstalling a different
pinned `torch` version risks breaking that match. So instead of blindly
`pip install torch==X`, this records exactly what's actually present
(and pins `numpy` only, which has no such GPU coupling), so the
environment is reproducible *as observed*, not guessed.

In [2]:
import sys
import subprocess

print("Python:", sys.version)

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import numpy as np
print("numpy:", np.__version__)

# Record the full environment actually in effect this session, for
# reproducibility -- not committed to the repo (ephemeral per-runtime),
# just written alongside this run's outputs. We deliberately do NOT
# force-install/downgrade numpy or torch here: Colab's preinstalled
# stack (jax, opencv, cupy, etc.) is already built against whatever
# numpy version ships by default, and reinstalling a different one
# mid-session breaks that instead of fixing anything.
with open("/content/environment-lock.txt", "w") as f:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=f)
print("Full environment snapshot written to /content/environment-lock.txt")

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
torch: 2.11.0+cu128 | CUDA available: True
GPU: Tesla T4
numpy: 2.1.3
Full environment snapshot written to /content/environment-lock.txt


## 3. Mount the corpus

`data/` is gitignored in the repo (raw corpora are never committed), so
`train.bin` / `train.bin.manifest.json` / `vocab.json` / `merges.txt`
(all produced by `python scripts/prepare_slm_corpus.py` — see that
script, not reinvented here) need to already exist somewhere reachable
from Colab:

- **Preferred:** upload the `prepare_slm_corpus.py` output directory to
  Google Drive once, then mount Drive here on every session.
- **Fallback:** no Drive copy yet? The next cell falls back to a direct
  file upload picker for the 4 files above.

Either way, set `CORPUS_DIR` below to wherever the 4 files end up.

In [3]:
from pathlib import Path

CORPUS_DIR = Path("/content/drive/MyDrive/adhan/data/final/tamil_slm")  # <-- adjust to your Drive path
REQUIRED_FILES = ["train.bin", "train.bin.manifest.json", "vocab.json", "merges.txt"]

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
except ImportError:
    print("Not running in Colab (or Drive unavailable) -- skipping mount.")

if not CORPUS_DIR.exists() or not all((CORPUS_DIR / f).exists() for f in REQUIRED_FILES):
    print(f"Corpus not found at {CORPUS_DIR} -- falling back to direct upload.")
    print(f"Please select all 4 files: {REQUIRED_FILES}")
    from google.colab import files

    CORPUS_DIR = Path("/content/uploaded_corpus")
    CORPUS_DIR.mkdir(exist_ok=True)
    uploaded = files.upload()
    for name, data in uploaded.items():
        (CORPUS_DIR / name).write_bytes(data)

missing = [f for f in REQUIRED_FILES if not (CORPUS_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Still missing {missing} in {CORPUS_DIR}. "
        "Generate them first with: python scripts/prepare_slm_corpus.py "
        "--corpus <your_corpus_dir> --out data/final/tamil_slm"
    )
print(f"Corpus ready at {CORPUS_DIR}: {REQUIRED_FILES}")


Mounted at /content/drive
Corpus ready at /content/drive/MyDrive/adhan/data/final/tamil_slm: ['train.bin', 'train.bin.manifest.json', 'vocab.json', 'merges.txt']


## 4. Load one real batch

Uses `read_shard()` + `load_manifest()` from
`src/adhan_slm/data/packing.py` -- the manifest-driven reader that
matches exactly how `prepare_slm_corpus.py` writes `train.bin`
(`n_sequences` rows of exactly `seq_len` tokens each, per the
`.manifest.json` sidecar).

**Why not `scripts/train_efficient.py`'s `MemmapShardDataset`?** That
class assumes each stored row is `seq_len + 1` tokens (one extra token
so it can slice `[:-1]` / `[1:]` into input/target pairs itself) and
derives the sequence count from raw file size alone, with no manifest.
That is a **different on-disk format** than what `packing.py` actually
writes (confirmed while building this notebook: a real `train.bin` of
3,632 tokens at `seq_len=16` is exactly `227 x 16`, not divisible into
`seq_len + 1 = 17`-token rows). Feeding a `prepare_slm_corpus.py` shard
into `MemmapShardDataset` would silently misalign every row instead of
raising an error. That mismatch is a separate, real issue against
`scripts/train_efficient.py` and is intentionally **not** patched here
-- this notebook sidesteps it entirely by using the reader that's
actually correct for this format, and stays within its own
"no model code" scope (constructing input/target pairs is a
training-loop concern for the next task, not this one).

In [4]:
import sys

sys.path.insert(0, f"{REPO_DIR}/src")

from adhan_slm.data.packing import load_manifest, read_shard  # noqa: E402

manifest = load_manifest(CORPUS_DIR / "train.bin")
print(f"manifest: seq_len={manifest.seq_len}, n_sequences={manifest.n_sequences}, "
      f"vocab_size={manifest.vocab_size}, dtype={manifest.dtype}")

shard = read_shard(CORPUS_DIR / "train.bin", manifest)
print(f"full shard array shape: {shard.shape}  (n_sequences, seq_len)")

BATCH_SIZE = 8
batch = shard[:BATCH_SIZE]
print(f"batch shape: {batch.shape}  (batch_size, seq_len)")
print(f"dtype: {batch.dtype}")


manifest: seq_len=128, n_sequences=1610, vocab_size=3000, dtype=<u2
full shard array shape: (1610, 128)  (n_sequences, seq_len)
batch shape: (8, 128)  (batch_size, seq_len)
dtype: uint16


## 5. Sanity check: decode one sequence back to Tamil

Not model code -- just proves the mounted corpus round-trips correctly
(right vocab, right byte order) rather than being garbled or
misaligned, before anyone builds a training loop on top of it.

In [5]:
from adhan_slm.tokenizer import SwaramTokenizer  # noqa: E402

tok = SwaramTokenizer.from_files(str(CORPUS_DIR / "vocab.json"), str(CORPUS_DIR / "merges.txt"))
decoded = tok.decode(batch[0].tolist())
print("Decoded first sequence in the batch:")
print(decoded)


Decoded first sequence in the batch:
திருக்குறள் 579:
ஒறுத்தாற்றும் பண்பினார் கண்ணும்கண் ணோடிப்
பொறுத்தாற்றும் பண்பே தலை.

மு. வரதராசனார் உரை: தண்டித்தற்குரிய தன்மை உடையவரிடத்திலும் கண்ணோட்டம் செய்து ( அவர் செய்த குற்றத்தைப்) பொருத்துக் காக்கும் பண்பே சிறந்தது.

சாலமன் பாப்பையா உரை: தம்மை வருத்தும் இயல்புடையவரிடத்திலும் கண்ணோட்டம் கொண்டு, அவர்தம் பிழையைப் பொறுக்கும் பண்பே சிறந்தது.

கலைஞர் உரை: அழிக்க நினைத்திடும் இயல்புடையவரிடத்திலும் பொறுமை காட்டுவது மிக உயர்ந்த பண்பாகும்திருக்குறள் 281:
எள்ளாமை


## Next steps (out of scope for this notebook)

Environment, corpus mount, and batch loading are all proven working
above, using the reader that's actually correct for this shard format.
Model definition and the training loop are the next task, built on top
of this scaffold rather than inside it -- and should settle the
`MemmapShardDataset` vs `packing.py` format question (flagged above)
before wiring PyTorch training to real `prepare_slm_corpus.py` output.